In [2]:
import requests
import json
import time

# Вариант 1: Используем веса модели + код архитектуры

In [6]:
def test_model_in_service():
    """Тестирование модели через сервис"""
    
    # Конфигурация тестирования
    # Крайне важно корректно заполнять и передавать все необходимые поля
    config = {
        "model_class_name": "LSTMPredictor",
        "target_feature": "close",
        "features": ["open", "high", "low", "close", "volume"],
        "problem_type": "regression",
        "normalization": "minmax",
        'sequence_length': 10 # размер батча
    }
    
    # Загружаем тестовые данные
    with open('test_data.json', 'r') as f:
        test_data = json.load(f)
    
    # URL сервиса тестирования (предполагается, что он запущен на порту 5001)
    url = "http://localhost:5001/api/models/test"
    
    print("Отправка модели на тестирование...")
    
    files = {
        'model': open('lstm_model_weights.pth', 'rb'),
        'model_code': open('model_architecture.py', 'rb')
    }
    
    data = {
        'config': json.dumps(config),
        'test_data': json.dumps(test_data)
    }
    
    try:
        response = requests.post(url, files=files, data=data)
        result = response.json()
        
        if response.status_code == 200:
            print("✅ Тестирование завершено успешно!")
            print(f"📊 Метрики модели:")
            for metric, value in result['metrics'].items():
                print(f"   {metric}: {value:.6f}")
            
            print(f"📈 Примеры предсказаний:")
            for i, pred in enumerate(result['predictions'][:5]):
                print(f"   {pred['timestamp']}: фактическое={pred['actual']:.2f}, предсказание={pred['predicted']:.2f}")
            
            # Сохраняем полный результат
            with open('test_results.json', 'w') as f:
                json.dump(result, f, indent=2)
            print("💾 Полный результат сохранен в test_results.json")
            
        else:
            print(f"❌ Ошибка тестирования: {result.get('error', 'Unknown error')}")
            
    except Exception as e:
        print(f"❌ Ошибка подключения к сервису: {e}")
        print("Убедитесь, что сервис тестирования запущен на localhost:5001")

# Вариант 2: Используем TorchScript модель

In [ ]:
def test_with_torchscript():
    """Тестирование TorchScript модели"""
    
    config = {
        "target_feature": "close",
        "features": ["open", "high", "low", "close", "volume"],
        "problem_type": "regression",
        'sequence_length': 10
    }
    
    with open('test_data.json', 'r') as f:
        test_data = json.load(f)
    
    url = "http://localhost:5001/api/models/test"
    
    # TorchScript не требует передачи кода модели
    files = {
        'model': open('lstm_model_scripted.pt', 'rb')
    }
    
    data = {
        'config': json.dumps(config),
        'test_data': json.dumps(test_data)
    }
    
    print("Тестирование TorchScript модели...")
    response = requests.post(url, files=files, data=data)
    result = response.json()
    
    if response.status_code == 200:
        print("✅ TorchScript модель протестирована успешно!")
        print("Метрики:", result['metrics'])
    else:
        print(f"❌ Ошибка: {result.get('error', 'Unknown error')}")

### HealthCheck

In [ ]:
def check_service_health():
    """Проверка доступности сервиса"""
    try:
        response = requests.get("http://localhost:5001/api/health")
        if response.status_code == 200:
            print("✅ Сервис тестирования доступен")
            return True
        else:
            print("❌ Сервис тестирования не отвечает")
            return False
    except:
        print("❌ Не удалось подключиться к сервису тестирования")
        return False

# Пример тестирования ML модели двумя способами

In [9]:
if check_service_health():
    print("\n1. Тестирование модели с кастомной архитектурой:")
    test_model_in_service()
    
    print("\n2. Тестирование TorchScript модели:")
    test_with_torchscript()

✅ Сервис тестирования доступен

1. Тестирование модели с кастомной архитектурой:
Отправка модели на тестирование...
✅ Тестирование завершено успешно!
📊 Метрики модели:
   mae: 106.460571
   mape: 99.873972
   mse: 11346.835938
   r2: -872.769409
   rmse: 106.521528
📈 Примеры предсказаний:
   2024-01-01 00:09:00: фактическое=104.59, предсказание=0.13
   2024-01-01 00:10:00: фактическое=104.78, предсказание=0.13
   2024-01-01 00:11:00: фактическое=104.10, предсказание=0.13
   2024-01-01 00:12:00: фактическое=104.46, предсказание=0.13
   2024-01-01 00:13:00: фактическое=103.91, предсказание=0.13
💾 Полный результат сохранен в test_results.json

2. Тестирование TorchScript модели:
Тестирование TorchScript модели...
✅ TorchScript модель протестирована успешно!
Метрики: {'mae': 106.4605712890625, 'mape': 99.87397193908691, 'mse': 11346.8359375, 'r2': -872.7694091796875, 'rmse': 106.52152804715111}
